# Exploration des sources TPCx-BB avec PySpark

TP Jour 3 – UC-06. Les fichiers `.dat` (séparateur `|`, sans en-tête) sont lus **en texte** avec les noms de colonnes du DDL de la spec TPCx-BB v1.7.0. Les fichiers sources ne sont jamais modifiés.

## 1. Environnement

In [1]:
import sys, platform, subprocess
sys.path.append('/work/scripts')
from commun import *
import pyspark
print('Python :', platform.python_version())
print('PySpark :', pyspark.__version__)
print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.splitlines()[0])
spark = session('exploration')
print('Spark :', spark.version, '| mode :', spark.sparkContext.master)
print('RAW :', RAW, '| sorties :', DATA)

Python : 3.12.14
PySpark : 4.3.0
openjdk version "21.0.13" 2026-10-20


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 14:24:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark : 4.3.0 | mode : local[4]
RAW : /data/raw | sorties : /work/data


## 2. Fonction d'exploration

Pour chaque source : schéma, échantillon, nombre de lignes, valeurs manquantes, doublons sur la clé candidate, min/max des colonnes numériques.

In [2]:
from pyspark.sql import functions as F

def explorer(table, cle):
    df = lire_raw(spark, table)
    print('=====', table, '=====')
    df.printSchema()
    df.show(5, truncate=False)
    n = df.count()
    print('Lignes :', n, '| colonnes :', len(df.columns))
    nulls = df.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).first().asDict()
    print('Valeurs manquantes (colonne : nb, %) :')
    for c, v in nulls.items():
        if v: print(f'  {c} : {v} ({100 * v / n:.3f} %)')
    if not any(nulls.values()): print('  aucune')
    distincts = df.select(*cle).distinct().count()
    print('Cle candidate', cle, ':', distincts, 'valeurs distinctes ->', 'doublons :', n - distincts)
    num = [c for c, t in SCHEMA[table] if t in ('bigint', 'int')]
    mm = df.agg(*[F.min(F.col(c).cast('long')).alias('min_' + c) for c in num] + [F.max(F.col(c).cast('long')).alias('max_' + c) for c in num]).first().asDict()
    print('Min / max numeriques :')
    for c in num: print(f'  {c} : {mm["min_" + c]} -> {mm["max_" + c]}')
    return df

## 3. Source 1 – customer (clients, une ligne par client)

In [3]:
customer = explorer('customer', ['c_customer_sk'])

===== customer =====
root
 |-- c_customer_sk: string (nullable = true)
 |-- c_customer_id: string (nullable = true)
 |-- c_current_cdemo_sk: string (nullable = true)
 |-- c_current_hdemo_sk: string (nullable = true)
 |-- c_current_addr_sk: string (nullable = true)
 |-- c_first_shipto_date_sk: string (nullable = true)
 |-- c_first_sales_date_sk: string (nullable = true)
 |-- c_salutation: string (nullable = true)
 |-- c_first_name: string (nullable = true)
 |-- c_last_name: string (nullable = true)
 |-- c_preferred_cust_flag: string (nullable = true)
 |-- c_birth_day: string (nullable = true)
 |-- c_birth_month: string (nullable = true)
 |-- c_birth_year: string (nullable = true)
 |-- c_birth_country: string (nullable = true)
 |-- c_login: string (nullable = true)
 |-- c_email_address: string (nullable = true)
 |-- c_last_review_date: string (nullable = true)



+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+------------+-----------+---------------------+-----------+-------------+------------+--------------------+------------+------------------------------+------------------+
|c_customer_sk|c_customer_id   |c_current_cdemo_sk|c_current_hdemo_sk|c_current_addr_sk|c_first_shipto_date_sk|c_first_sales_date_sk|c_salutation|c_first_name|c_last_name|c_preferred_cust_flag|c_birth_day|c_birth_month|c_birth_year|c_birth_country     |c_login     |c_email_address               |c_last_review_date|
+-------------+----------------+------------------+------------------+-----------------+----------------------+---------------------+------------+------------+-----------+---------------------+-----------+-------------+------------+--------------------+------------+------------------------------+------------------+
|0            |AAAAAAAAAAAAAAAA|1824793          

Lignes : 99000 | colonnes : 18


Valeurs manquantes (colonne : nb, %) :
  c_current_cdemo_sk : 18 (0.018 %)
  c_current_hdemo_sk : 29 (0.029 %)
  c_current_addr_sk : 22 (0.022 %)
  c_first_shipto_date_sk : 25 (0.025 %)
  c_first_sales_date_sk : 18 (0.018 %)
  c_salutation : 28 (0.028 %)
  c_preferred_cust_flag : 27 (0.027 %)
  c_birth_day : 18 (0.018 %)
  c_birth_month : 25 (0.025 %)
  c_birth_year : 30 (0.030 %)
  c_birth_country : 16 (0.016 %)
  c_login : 26 (0.026 %)
  c_email_address : 23 (0.023 %)
  c_last_review_date : 28 (0.028 %)


Cle candidate ['c_customer_sk'] : 99000 valeurs distinctes -> doublons : 0


Min / max numeriques :
  c_customer_sk : 0 -> 98999
  c_current_cdemo_sk : 18 -> 1920796
  c_current_hdemo_sk : 0 -> 7199
  c_current_addr_sk : 0 -> 49499
  c_first_shipto_date_sk : 3 -> 109570
  c_first_sales_date_sk : 1 -> 109568
  c_birth_day : 1 -> 31
  c_birth_month : 1 -> 12
  c_birth_year : 1924 -> 1992


## 4. Source 2 – customer_address (adresses, une ligne par adresse)

In [4]:
adresse = explorer('customer_address', ['ca_address_sk'])
adresse.groupBy('ca_state').count().orderBy(F.desc('count')).show(5)

===== customer_address =====
root
 |-- ca_address_sk: string (nullable = true)
 |-- ca_address_id: string (nullable = true)
 |-- ca_street_number: string (nullable = true)
 |-- ca_street_name: string (nullable = true)
 |-- ca_street_type: string (nullable = true)
 |-- ca_suite_number: string (nullable = true)
 |-- ca_city: string (nullable = true)
 |-- ca_county: string (nullable = true)
 |-- ca_state: string (nullable = true)
 |-- ca_zip: string (nullable = true)
 |-- ca_country: string (nullable = true)
 |-- ca_gmt_offset: string (nullable = true)
 |-- ca_location_type: string (nullable = true)



+-------------+----------------+----------------+--------------+--------------+---------------+-----------+-------------------+--------+------+-------------+-------------+----------------+
|ca_address_sk|ca_address_id   |ca_street_number|ca_street_name|ca_street_type|ca_suite_number|ca_city    |ca_county          |ca_state|ca_zip|ca_country   |ca_gmt_offset|ca_location_type|
+-------------+----------------+----------------+--------------+--------------+---------------+-----------+-------------------+--------+------+-------------+-------------+----------------+
|0            |AAAAAAAAAAAAAAAA|561             |Cedar 12th    |Road          |J5jix6JEFD     |Genoa      |Tuscarawas County  |OH      |64453 |United States|-5           |single family   |
|1            |AAAAAAAAAAAAAAAB|473             |Birch Poplar  |Ct.           |tEmwYrz        |Chapel Hill|Hodgeman County    |KS      |17270 |United States|-6           |single family   |
|2            |AAAAAAAAAAAAAAAC|786             |2nd Po

Lignes : 49500 | colonnes : 13


Valeurs manquantes (colonne : nb, %) :
  ca_street_number : 14 (0.028 %)
  ca_street_name : 10 (0.020 %)
  ca_street_type : 13 (0.026 %)
  ca_suite_number : 23 (0.046 %)
  ca_zip : 14 (0.028 %)


Cle candidate ['ca_address_sk'] : 49500 valeurs distinctes -> doublons : 0


Min / max numeriques :
  ca_address_sk : 0 -> 49499


+--------+-----+
|ca_state|count|
+--------+-----+
|      TX| 4069|
|      GA| 2539|
|      VA| 2168|
|      KY| 1927|
|      KS| 1685|
+--------+-----+
only showing top 5 rows


## 5. Source 3 – household_demographics (foyers)

In [5]:
foyer = explorer('household_demographics', ['hd_demo_sk'])
foyer.groupBy('hd_vehicle_count').count().orderBy('hd_vehicle_count').show()

===== household_demographics =====
root
 |-- hd_demo_sk: string (nullable = true)
 |-- hd_income_band_sk: string (nullable = true)
 |-- hd_buy_potential: string (nullable = true)
 |-- hd_dep_count: string (nullable = true)
 |-- hd_vehicle_count: string (nullable = true)

+----------+-----------------+----------------+------------+----------------+
|hd_demo_sk|hd_income_band_sk|hd_buy_potential|hd_dep_count|hd_vehicle_count|
+----------+-----------------+----------------+------------+----------------+
|0         |3                |1001-5000       |0           |0               |
|1         |12               |5001-10000      |1           |2               |
|2         |9                |0-500           |2           |3               |
|3         |7                |501-1000        |7           |2               |
|4         |16               |1001-5000       |6           |2               |
+----------+-----------------+----------------+------------+----------------+
only showing top 5 rows


Lignes : 7200 | colonnes : 5


Valeurs manquantes (colonne : nb, %) :
  hd_income_band_sk : 2 (0.028 %)
  hd_buy_potential : 1 (0.014 %)
  hd_dep_count : 2 (0.028 %)
  hd_vehicle_count : 2 (0.028 %)
Cle candidate ['hd_demo_sk'] : 7200 valeurs distinctes -> doublons : 0


Min / max numeriques :
  hd_demo_sk : 0 -> 7199
  hd_income_band_sk : 0 -> 19
  hd_dep_count : 0 -> 9
  hd_vehicle_count : -1 -> 4
+----------------+-----+
|hd_vehicle_count|count|
+----------------+-----+
|            NULL|    2|
|              -1|  282|
|               0| 1183|
|               1| 1243|
|               2| 2514|
|               3| 1596|
|               4|  380|
+----------------+-----+



## 6. Source 4 – income_band (tranches de revenu)

In [6]:
tranche = explorer('income_band', ['ib_income_band_sk'])

===== income_band =====
root
 |-- ib_income_band_sk: string (nullable = true)
 |-- ib_lower_bound: string (nullable = true)
 |-- ib_upper_bound: string (nullable = true)

+-----------------+--------------+--------------+
|ib_income_band_sk|ib_lower_bound|ib_upper_bound|
+-----------------+--------------+--------------+
|0                |1             |10000         |
|1                |10001         |20000         |
|2                |20001         |30000         |
|3                |30001         |40000         |
|4                |40001         |50000         |
+-----------------+--------------+--------------+
only showing top 5 rows
Lignes : 20 | colonnes : 3


Valeurs manquantes (colonne : nb, %) :
  aucune


Cle candidate ['ib_income_band_sk'] : 20 valeurs distinctes -> doublons : 0
Min / max numeriques :
  ib_income_band_sk : 0 -> 19
  ib_lower_bound : 1 -> 190001
  ib_upper_bound : 10000 -> 200000


## 7. Source 5 – date_dim (calendrier)

In [7]:
dates = explorer('date_dim', ['d_date_sk'])
dates.select(F.min('d_date'), F.max('d_date')).show()

===== date_dim =====
root
 |-- d_date_sk: string (nullable = true)
 |-- d_date_id: string (nullable = true)
 |-- d_date: string (nullable = true)
 |-- d_month_seq: string (nullable = true)
 |-- d_week_seq: string (nullable = true)
 |-- d_quarter_seq: string (nullable = true)
 |-- d_year: string (nullable = true)
 |-- d_dow: string (nullable = true)
 |-- d_moy: string (nullable = true)
 |-- d_dom: string (nullable = true)
 |-- d_qoy: string (nullable = true)
 |-- d_fy_year: string (nullable = true)
 |-- d_fy_quarter_seq: string (nullable = true)
 |-- d_fy_week_seq: string (nullable = true)
 |-- d_day_name: string (nullable = true)
 |-- d_quarter_name: string (nullable = true)
 |-- d_holiday: string (nullable = true)
 |-- d_weekend: string (nullable = true)
 |-- d_following_holiday: string (nullable = true)
 |-- d_first_dom: string (nullable = true)
 |-- d_last_dom: string (nullable = true)
 |-- d_same_day_ly: string (nullable = true)
 |-- d_same_day_lq: string (nullable = true)
 |-- d_c

Lignes : 109573 | colonnes : 28


Valeurs manquantes (colonne : nb, %) :
  aucune


Cle candidate ['d_date_sk'] : 109573 valeurs distinctes -> doublons : 0


Min / max numeriques :
  d_date_sk : 0 -> 109572
  d_month_seq : 0 -> 3599
  d_week_seq : 0 -> 15653
  d_quarter_seq : 0 -> 1199
  d_year : 1900 -> 2199
  d_dow : 1 -> 7
  d_moy : 1 -> 12
  d_dom : 1 -> 31
  d_qoy : 1 -> 4
  d_fy_year : 1900 -> 2199
  d_fy_quarter_seq : 0 -> 1199
  d_fy_week_seq : 0 -> 15653
  d_first_dom : 2415021 -> 2488069
  d_last_dom : 2415020 -> 2488370
  d_same_day_ly : 2414657 -> 2487705
  d_same_day_lq : 2414930 -> 2487977


+-----------+-----------+
|min(d_date)|max(d_date)|
+-----------+-----------+
| 1900-01-01| 2199-12-31|
+-----------+-----------+



## 8. Source 6 – inventory (stocks, une ligne par date, article et entrepôt)

Le fichier est tronqué : on s'attend à une dernière ligne incomplète.

In [8]:
inv = explorer('inventory', ['inv_date_sk', 'inv_item_sk', 'inv_warehouse_sk'])
print('Lignes dont une cle est absente (ligne incomplete) :')
inv.filter(F.col('inv_item_sk').isNull() | F.col('inv_warehouse_sk').isNull()).show(truncate=False)
print('Lignes par date (5 dernieres dates) :')
inv.groupBy('inv_date_sk').count().orderBy(F.desc('inv_date_sk')).show(5)

===== inventory =====
root
 |-- inv_date_sk: string (nullable = true)
 |-- inv_item_sk: string (nullable = true)
 |-- inv_warehouse_sk: string (nullable = true)
 |-- inv_quantity_on_hand: string (nullable = true)

+-----------+-----------+----------------+--------------------+
|inv_date_sk|inv_item_sk|inv_warehouse_sk|inv_quantity_on_hand|
+-----------+-----------+----------------+--------------------+
|36890      |0          |0               |503                 |
|36890      |1          |0               |624                 |
|36890      |2          |0               |796                 |
|36890      |3          |0               |229                 |
|36890      |4          |0               |699                 |
+-----------+-----------+----------------+--------------------+
only showing top 5 rows


Lignes : 21791809 | colonnes : 4


Valeurs manquantes (colonne : nb, %) :
  inv_warehouse_sk : 1 (0.000 %)
  inv_quantity_on_hand : 5617 (0.026 %)


Cle candidate ['inv_date_sk', 'inv_item_sk', 'inv_warehouse_sk'] : 21791809 valeurs distinctes -> doublons : 0


Min / max numeriques :
  inv_date_sk : 36890 -> 38584
  inv_item_sk : 0 -> 17999
  inv_warehouse_sk : 0 -> 4
  inv_quantity_on_hand : 0 -> 1000
Lignes dont une cle est absente (ligne incomplete) :


+-----------+-----------+----------------+--------------------+
|inv_date_sk|inv_item_sk|inv_warehouse_sk|inv_quantity_on_hand|
+-----------+-----------+----------------+--------------------+
|38584      |1180       |NULL            |NULL                |
+-----------+-----------+----------------+--------------------+

Lignes par date (5 dernieres dates) :


+-----------+-----+
|inv_date_sk|count|
+-----------+-----+
|      38584|11809|
|      38577|90000|
|      38570|90000|
|      38563|90000|
|      38556|90000|
+-----------+-----+
only showing top 5 rows


## 9. Constats

- Les 6 sources sont **structurées** (texte délimité par `|`, CRLF, sans en-tête) ; les types viennent du DDL de la spec, pas du fichier.
- Clés candidates uniques sur les 6 sources ; aucun doublon sur la clé.
- Valeurs manquantes : environ 0,02 à 0,05 % dans `customer` et `customer_address`, `inv_quantity_on_hand` à 0,026 %, rien dans `date_dim` ni `income_band`.
- `hd_vehicle_count` contient la valeur **-1** (non conforme à un nombre de véhicules).
- `inventory` : dernière ligne incomplète (fichier tronqué) et dernière date à 11 808 lignes au lieu de 90 000.
- `date_dim` couvre 1900 à 2199 (109 573 jours).
- Les décisions de nettoyage sont dans la partie « Règles de nettoyage » de `UC06_traitements_PySpark.pdf` et appliquées par `scripts/nettoyage.py`.

In [9]:
spark.stop()